In [7]:
# Importamos los módulos
import certifi
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import urllib3
import requests
from urllib3 import request
from unicodedata import normalize
from io import StringIO

### Formato.parquet

In [6]:
# Es necesario instalar pyarrow para poder leer archivos Parquet. Se puede instalar con el siguiente comando:
# uv add pyarrow
# Leemos la data desde el archivo Parquet. Usamos este método (parquet) para tener una mayor velocidad de procesado.
# https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page
df_parquet = pd.read_parquet("archivos/yellow_tripdata_2026-01.parquet")
df_parquet.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


### Formato .csv

In [5]:
# Usaremos el archivo de este URL: https://data.cityofnewyork.us/resource/h9gi-nx95.csv?$limit=500
df_csv = pd.read_csv("archivos/h9gi-nx95.csv")
# Nos muestra los primeros 5 registros del archivo CSV
df_csv.head()

,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,off_street_name,cross_street_name,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code1,vehicle_type_code2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,2021-09-11T00:00:00.000,2:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
1,2022-03-26T00:00:00.000,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,...,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN
2,2023-11-01T00:00:00.000,1:29,BROOKLYN,11230.0,40.62179,-73.970024,"\n, \n(40.62179, -73.970024)",OCEAN PARKWAY,AVENUE K,NaN,...,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
3,2022-06-29T00:00:00.000,6:55,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
4,2022-09-21T00:00:00.000,13:21,NaN,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4566131,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN


### Obteniendo data desde una API

In [ ]:
# Importamos los módulos necesarios para hacer la solicitud HTTP a la API de NYC Open Data
import pandas as pd
import json
import certifi
import urllib3

# Trabajaremos con la siguiente URL, está restringida para descargar 500 registros
url = 'https://data.cityofnewyork.us/resource/h9gi-nx95.json?$limit=500'

# Creamos un PoolManager para manejar las solicitudes HTTP y verificar que el certificado SSL es válido.
http = urllib3.PoolManager(cert_reqs='CERT_REQUIRED', ca_certs=certifi.where())

# Hacemos la solicitud HTTP a la URL y obtenemos la respuesta.
response = http.request('GET', url)
apt_status = response.status
print(apt_status)

# El estatus 200 nos indica que la API está disponible y podemos leer los datos.
if apt_status == 200:
    data = json.loads(response.data.decode('utf-8'))
    df_api = pd.json_normalize(data)
else:
    print(f'Error al obtener datos: código {apt_status}')
    df_api = pd.DataFrame()

df_api.head(10)

200


,crash_date,crash_time,on_street_name,off_street_name,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,...,contributing_factor_vehicle_3,vehicle_type_code_3,location.latitude,location.longitude,location.human_address,cross_street_name,contributing_factor_vehicle_4,vehicle_type_code_4,contributing_factor_vehicle_5,vehicle_type_code_5
0,2021-09-11T00:00:00.000,2:39,WHITESTONE EXPRESSWAY,20 AVENUE,2,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-03-26T00:00:00.000,11:45,QUEENSBORO BRIDGE UPPER,NaN,1,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-11-01T00:00:00.000,1:29,OCEAN PARKWAY,AVENUE K,1,0,0,0,0,0,...,Unspecified,Sedan,40.62179,-73.970024,"{""address"": """", ""city"": """", ""state"": """", ""zip""...",NaN,NaN,NaN,NaN,NaN
3,2022-06-29T00:00:00.000,6:55,THROGS NECK BRIDGE,NaN,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-09-21T00:00:00.000,13:21,BROOKLYN BRIDGE,NaN,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2023-04-26T00:00:00.000,13:30,WEST 54 STREET,NaN,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2023-11-01T00:00:00.000,7:12,HUTCHINSON RIVER PARKWAY,NaN,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2023-11-01T00:00:00.000,8:01,WEST 35 STREET,HENRY HUDSON RIVER,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2023-04-26T00:00:00.000,22:20,NaN,NaN,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,61 Ed Koch queensborough bridge,NaN,NaN,NaN,NaN
9,2021-09-11T00:00:00.000,9:35,NaN,NaN,0,0,0,0,0,0,...,NaN,NaN,40.667202,-73.8665,"{""address"": """", ""city"": """", ""state"": """", ""zip""...",1211 LORING AVENUE,NaN,NaN,NaN,NaN


### Obteniendo data de las tablas RDBMS / una base de datos relacional como postgreSQL

In [9]:
# Importamos los módulos necesarios para hacer la solicitud HTTP a la API de NYC Open Data
import pandas as pd
import sqlite3

# Lee las resultados del sqllite (paquete sql ligero en el entorno) dentro de una tabla de pandas.
with sqlite3.connect("archivos/movies.sqlite") as conn:
    df = pd.read_sql("SELECT * from movies", conn)
df.head()

,id,original_title,budget,popularity,release_date,revenue,title,vote_average,vote_count,overview,tagline,uid,director_id
0,43597,Avatar,237000000,150,2009-12-10,2787965087,Avatar,7.2,11800,"In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,19995,4762
1,43598,Pirates of the Caribbean: At World's End,300000000,139,2007-05-19,961000000,Pirates of the Caribbean: At World's End,6.9,4500,"Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",285,4763
2,43599,Spectre,245000000,107,2015-10-26,880674609,Spectre,6.3,4466,A cryptic message from Bond’s past sends him o...,A Plan No One Escapes,206647,4764
3,43600,The Dark Knight Rises,250000000,112,2012-07-16,1084939099,The Dark Knight Rises,7.6,9106,Following the death of District Attorney Harve...,The Legend Ends,49026,4765
4,43601,John Carter,260000000,43,2012-03-07,284139100,John Carter,6.1,2124,"John Carter is a war-weary, former military ca...","Lost in our world, found in another.",49529,4766


### Obteniendo datos de una página web

In [22]:
import requests
import pandas as pd
from io import StringIO

url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'

response = requests.get(
    url,
    headers={'User-Agent': 'Mozilla/5.0'},
    timeout=30
)
response.raise_for_status()  # Lanza excepción si falla

# Leemos las tablas HTML de la página
df_html = pd.read_html(StringIO(response.text), match='by country')

print(len(df_html))  # cuántas tablas coinciden
df_html[0]

4


,Country/Territory,IMF (2026)[1],World Bank (2025)[6],United Nations (2024)[7]
0,World,126295331,118350166,100834796
1,United States,32383920,30769700,29298000
2,China[n 1],20851593,19498039,18743802
3,Germany,5452858,5050923,4659929
4,Japan,4379253,4435163,4026211
...,...,...,...,...
217,Palau,377,345,310
218,Marshall Islands,342,308,281
219,Nauru,196,176,187
220,Montserrat,—N/a,—N/a,81


## Pipeline de extracción
MÓDULO DE EXTRACCIÓN DE DATOS (ETAPA "EXTRACT" DE UN PIPELINE ETL)

Este script reúne 5 funciones, cada una encargada de traer datos desde una fuente distinta (parquet, csv, API, base de datos SQLite, página web) y devolverlos en un DataFrame de pandas, que servirá para los siguientes pasos del pipeline de otros módulos.

Nos aseguramos de que los paquetes estén instalados e importamos los módulos necesarios.
uv add pyarrow      -> necesario para que pandas pueda leer archivos .parquet
uv add certifi      -> nos da la lista de certificados SSL válidos y actualizados

In [ ]:
import urllib3       # Librería para hacer solicitudes HTTP (usada para la API)
import certifi       # Provee los certificados SSL para validar conexiones HTTPS seguras
import json          # Para convertir texto JSON (de la API) en estructuras de Python
import sqlite3       # Para conectarnos a bases de datos SQLite
import pandas as pd  # La librería principal: todo termina convertido en DataFrame
import logging       # Para registrar mensajes (info, errores) en vez de usar print()
import requests      # Para descargar páginas web y leer su contenido
import urllib.request
from io import StringIO #  # Para convertir texto en un "stream" que pandas puede leer como si fuera un archivo
from pathlib import Path

# Configuración del logging
# __name__ toma el nombre del archivo/módulo actual. Esto permite que, si este script se importa desde otro archivo, los mensajes de log indiquen de qué
# módulo vienen (por ejemplo "data_extraction"), facilitando la depuración.
logger = logging.getLogger(__name__)

# FUNCIÓN 1: Extraer datos desde un archivo Parquet
def source_data_from_parquet(parquet_file_name):
    try:
        # Abrimos el archivo y lo cargamos en la memoria
        df_parquet = pd.read_parquet(parquet_file_name)
        # Registramos en el log cuántas filas (registros) se extrajeron con éxito.
        logger.info(f'{parquet_file_name} : extracted {df_parquet.shape[0]} records from the parquet file')
    except Exception as e:
        # Si algo falla (archivo no existe, está corrupto, etc.) No dejamos que el programa se caiga. En vez de eso, registramos el error con detalle
        # (logger.exception guarda automáticamente el traceback completo) y devolvemos un DataFrame vacío para que el pipeline pueda seguir.
        logger.exception(f'{parquet_file_name} : - exception {e} encountered while extracting the parquet file')
        df_parquet = pd.DataFrame()
    return df_parquet

# FUNCIÓN 2: Extraer datos desde un archivo CSV
def source_data_from_csv(csv_file_name):
    try:
        # pd.read_csv() abre el archivo de texto y lo interpreta como tabla
        df_csv = pd.read_csv(csv_file_name)
        logger.info(f'{csv_file_name} : extracted {df_csv.shape[0]} records from the csv file')
    except Exception as e:
        # Mismo patrón que la función anterior: nunca dejamos que un error detenga todo el pipeline, solo lo registramos.
        logger.exception(f'{csv_file_name} : - exception {e} encountered while extracting the csv_file_name file')
        df_csv = pd.DataFrame()
    return df_csv

# FUNCIÓN 3: Extraer datos desde una API REST (formato JSON)
def source_data_from_api(api_endpoint):
    """
    Hace una solicitud HTTP GET a una API que devuelve datos en formato JSON, y los convierte en un DataFrame.
    """
    # Inicializamos apt_status en None antes del try, porque si http.request() falla (por ejemplo, no hay conexión a internet), la excepción ocurriría 
    # antes de que la variable apt_status llegue a existir. Si luego intentamos usarla en el except sin haberla definido antes, Python lanzaría un 
    # NameError adicional. Definirla aquí evita ese problema.
    apt_status = None

    try:
        # Creamos un "administrador de conexiones" (PoolManager) que sabe cómo reutilizar conexiones HTTP de forma eficiente cert_reqs='CERT_REQUIRED' 
        # obliga a validar el certificado SSL del servidor (nunca se debe desactivar esta validación, es una medida de seguridad importante).
        # ca_certs=certifi.where() nos dice de dónde sacar la lista de certificados confiables.
        http = urllib3.PoolManager(cert_reqs='CERT_REQUIRED', ca_certs=certifi.where())
        # Hacemos la solicitud GET a la URL de la API
        api_response = http.request('GET', api_endpoint)
        # El código de estado HTTP nos dice si la solicitud fue exitosa 200 = OK. Otros códigos comunes: 404 = no encontrado, 500 = error del servidor.
        apt_status = api_response.status

        if apt_status == 200:
            logger.info(f'{apt_status} - ok : while invoking the api {api_endpoint}')
            # api_response.data son los bytes crudos de la respuesta .decode('utf-8') los convierte en texto legible.
            # json.loads() convierte ese texto JSON en listas/diccionarios de Python.
            data = json.loads(api_response.data.decode('utf-8'))
            # pd.json_normalize() convierte esa estructura (potencialmente anidada) en una tabla plana, ideal para trabajar con pandas.
            df_api = pd.json_normalize(data)
            logger.info(f'{apt_status}- extracted {df_api.shape[0]} records from the api')
        else:
            # Si el código de estado NO es 200, no intentamos leer los datos (probablemente no vienen en el formato esperado). 
            # Registramos el error y devolvemos un DataFrame vacío.
            logger.error(f'{apt_status}- error : while invoking the api {api_endpoint}')
            df_api = pd.DataFrame()  # OJO: DataFrame con "F" mayúscula, pd.Dataframe() no existe
    except Exception as e:
        # Aquí caen errores de conexión, timeouts, JSON mal formado, etc.
        logger.exception(f'{apt_status} : - exception {e} encountered while reading data from the api')
        df_api = pd.DataFrame()
    return df_api

# FUNCIÓN 4: Extraer datos desde una tabla de una base de datos SQLite
def source_data_from_table(db_name, table_name):
    try:
        # 'with sqlite3.connect(db_name) as conn' abre la conexión y garantiza que se cierre automáticamente al terminar el bloque, incluso si
        # ocurre un error dentro de él (esto se llama "context manager").
        with sqlite3.connect(db_name) as conn:
            # pd.read_sql() ejecuta la consulta SQL y devuelve el resultado directamente como DataFrame. Aquí usamos un f-string para
            # insertar el nombre de la tabla en la consulta.
            df_table = pd.read_sql(f"SELECT * from {table_name}", conn)
            logger.info(f'{db_name}- read {df_table.shape[0]} records from the table: {table_name}')
    except Exception as e:
        # Si la base de datos no existe, la tabla no existe, o hay un error de sintaxis SQL, lo registramos aquí sin detener el pipeline.
        logger.exception(f'{db_name} : - exception {e} encountered while reading data from the table: {table_name}')
        df_table = pd.DataFrame()
    return df_table

# FUNCIÓN 5: Extraer una tabla desde una página web (HTML)

def source_data_from_webpage(web_page_url, matching_keyword):
    """
    Descarga una página web usando un User-Agent personalizado para evitar bloqueos HTTP 403
    y extrae la primera tabla que coincida con la palabra clave.
    """
    try:
        # Creamos una petición asignando un User-Agent de navegador
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        req = urllib.request.Request(web_page_url, headers=headers)
        
        # Abrimos la URL con la petición configurada
        with urllib.request.urlopen(req) as response:
            html_content = response.read().decode('utf-8')
        
        # Le pasamos el contenido HTML directamente a pandas
        df_html_list = pd.read_html(html_content, match=matching_keyword)
        df_html = df_html_list[0]
        
        logger.info(f'{web_page_url} - read {df_html.shape[0]} records from the page')
    except Exception as e:
        logger.exception(f'{web_page_url} : - exception {e} encountered while reading data from the page')
        df_html = pd.DataFrame()
        
    return df_html


# FUNCIÓN ORQUESTADORA: llama a las 5 fuentes y devuelve todo junto
def extract_data():
    """
    Define la configuración de cada fuente (nombres de archivo, endpoint, base de datos, URL) y llama a las 5 funciones de extracción.
    Devuelve una tupla con los 5 DataFrames resultantes.
    """
    # Obtiene la ruta absoluta de la carpeta donde se encuentra este archivo .py
    BASE_DIR = Path(__file__).resolve().parent
    # --- Configuración de cada fuente ---
    parquet_file_name = BASE_DIR / "archivos" / "yellow_tripdata_2026-01.parquet"
    csv_file_name = BASE_DIR / "archivos" / "h9gi-nx95.csv"
    api_endpoint = "https://data.cityofnewyork.us/resource/h9gi-nx95.json?$limit=500"
    db_name = BASE_DIR / "archivos" / "movies.sqlite"
    table_name = "movies"
    web_page_url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
    matching_keyword = "by country"

    # Extraemos los datos de TODAS las fuentes.
    # Estos DataFrames quedarán disponibles para cargarlos en una tabla VSA, una tabla PSA, o para usarlos en un pipeline de transformación posterior.
    df_parquet, df_csv, df_api, df_table, df_html = (
        source_data_from_parquet(parquet_file_name),
        source_data_from_csv(csv_file_name),
        source_data_from_api(api_endpoint),
        source_data_from_table(db_name, table_name),
        source_data_from_webpage(web_page_url, matching_keyword),
    )

    return df_parquet, df_csv, df_api, df_table, df_html

# ============================================================================
# PUNTO DE ENTRADA: solo se ejecuta si corremos este archivo directamente
# (no se ejecuta si el archivo es importado desde otro script)
# ============================================================================
if __name__ == "__main__":
    # Configuramos el logging para que se muestre en consola con nivel INFO (así vemos los mensajes logger.info() y logger.error() al ejecutar el script)
    logging.basicConfig(level=logging.INFO)
    df_parquet, df_csv, df_api, df_table, df_html = extract_data()

    print("Vista previa de datos extraídos desde la API:")
    print(df_api.head())

INFO:__main__:archivos/yellow_tripdata_2026-01.parquet : extracted 3724889 records from the parquet file
INFO:__main__:archivos/h9gi-nx95.csv : extracted 500 records from the csv file
INFO:__main__:200 - ok : while invoking the api https://data.cityofnewyork.us/resource/h9gi-nx95.json?$limit=500
INFO:__main__:200- extracted 500 records from the api
INFO:__main__:archivos/movies.sqlite- read 4773 records from the table: movies
INFO:__main__:https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)- read 222 records from the page: https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)


Vista previa de datos extraídos desde la API:
                crash_date crash_time           on_street_name  \
0  2021-09-11T00:00:00.000       2:39    WHITESTONE EXPRESSWAY   
1  2022-03-26T00:00:00.000      11:45  QUEENSBORO BRIDGE UPPER   
2  2023-11-01T00:00:00.000       1:29            OCEAN PARKWAY   
3  2022-06-29T00:00:00.000       6:55       THROGS NECK BRIDGE   
4  2022-09-21T00:00:00.000      13:21          BROOKLYN BRIDGE   

  off_street_name number_of_persons_injured number_of_persons_killed  \
0       20 AVENUE                         2                        0   
1             NaN                         1                        0   
2        AVENUE K                         1                        0   
3             NaN                         0                        0   
4             NaN                         0                        0   

  number_of_pedestrians_injured number_of_pedestrians_killed  \
0                             0                            0